In [ ]:
_repo_root = !git rev-parse --show-toplevel
%cd {_repo_root[0]}
del _repo_root

# HIGGS scaling

Reader-facing companion for the dense HIGGS repeated-shuffle scaling experiment. It validates raw result completeness, performs validation-only learning-rate selection, and compares parameter-matched spectral and constant-width MLP families. It makes no empirical claim until a complete result file has been loaded and the cells have run.

## Setup

In [ ]:
import warnings

import matplotlib as mpl
import numpy as np
import pandas as pd

from paper.experiments.higgs_scaling import (
    PROFILES,
    default_raw_path,
    select_lr,
    summarize_raw,
    validate_raw,
)
from paper.plotting import (
    plot_higgs_models_by_dimension,
    plot_higgs_spectral_dimensions,
)

mpl.rcParams["figure.dpi"] = 140

In [ ]:
profile_name = "full"
variant = None  # Or one model name to inspect a single-family shard.
profile = PROFILES[profile_name]
raw_path = default_raw_path(profile_name, variant)
metric = "logloss"  # Or "brier".

## Context and assumptions

- Rows 0–9,999,999 train the models, the next 500,000 form validation, and the official final 500,000 rows form test. File order is not time.
- Means and standard deviations are fitted once on the complete training partition and then fixed for every checkpoint and model.
- A trajectory consumes independent shuffled passes over the fixed 10,000,000-row training pool. The x-axis is cumulative examples seen by the optimizer, not independently fitted dataset size.
- The full profile includes the requested power-of-two grid through `2**24`, a runtime checkpoint at the first complete pass (rounded forward to a global minibatch boundary), and the exact two-pass endpoint at 20,000,000 examples. Global minibatches may cross a pass boundary.
- Validation log loss at the final checkpoint selects one learning rate per model family and capacity. Evaluation retrains once with that rate and reports all checkpoints from the same trajectory.
- Lines are medians across paired data-order and initialization seeds; bands are interquartile seed variation on one fixed test partition, not confidence intervals for a test population.

## Load and validate results

In [ ]:
if not raw_path.exists():
    raise FileNotFoundError(
        f"{raw_path} does not exist; run paper.experiments.higgs_scaling first"
    )

raw = pd.read_csv(raw_path)
validate_raw(raw, profile, variant)

## Validation selection and recorded capacity

In [ ]:
selected = select_lr(raw)
selected_lrs = selected[["model", "dim", "selected_lr"]].drop_duplicates()
boundaries = selected_lrs.loc[
    np.isclose(selected_lrs["selected_lr"], min(profile.lrs))
    | np.isclose(selected_lrs["selected_lr"], max(profile.lrs))
]
if not boundaries.empty:
    warnings.warn(
        "selected learning rate touches the tuning-grid boundary for:\n"
        + boundaries.to_string(index=False),
        stacklevel=1,
    )

capacity_table = (
    selected[["model", "dim", "width", "num_parameters"]]
    .drop_duplicates()
    .sort_values(["dim", "model"], kind="stable")
    .reset_index(drop=True)
)
capacity_table

## Selected test summary

In [ ]:
summary = summarize_raw(raw)
summary[[
    "train_pool_size", "train_size", "model", "dim", "width", "num_parameters",
    "selected_lr", "median_test_logloss", "q25_test_logloss",
    "q75_test_logloss", "median_test_brier", "q25_test_brier",
    "q75_test_brier", "n",
]].sort_values(["train_size", "dim", "model"], kind="stable")

## Scaling curves

The matched-family plot requires the merged result rather than an individual variant shard. Facet annotations are read from `width` and `num_parameters` in the selected rows; they are not reconstructed from a lookup table.

In [ ]:
if variant is None:
    plot_higgs_models_by_dimension(selected, metric=metric)
else:
    print(
        "Matched-family facets require the merged result; "
        "set variant=None after appending all family shards."
    )

### Spectral dimensions on one axis

In [ ]:
if variant in (None, "spectral"):
    plot_higgs_spectral_dimensions(selected, metric=metric)
else:
    print(
        "Spectral-dimension comparison requires the merged result or "
        "the spectral shard."
    )

## Reading the figure

Each facet holds the spectral dimension—and therefore the target parameter budget—fixed. The linear trajectory is repeated as a common reference. The annotation reports the actual spectral parameter count and each MLP's recorded depth, width, and parameter count. The one-axis spectral plot compares the matrix dimensions directly against cumulative examples seen, including repeated passes over the recorded `train_pool_size`. Change `metric` in Setup to `"brier"` for the corresponding Brier-score view. Any substantive empirical interpretation should be written only after this notebook has executed against the complete result file.